# Preparing dataframes for pseudobulking analysis - Sample_type_major2
## Using raw anndata object NRclean_Raw_Data.h5ad 
* sum by sample, so for each DC2 subset (7 dataframes) made a dataframe that has genes as rows and integration_id (samples) as columns
* then make a metadata dataframe that has two columns --> integration_id and cancer type
* done on 19/10/26
* Need to use raw data when pseudobulking (not using 20250629_DC2_bbknn_clustered2.h5ad )

### Using code from NR 2025-01-13_Pseudobulking.ipynb


In [1]:
#load packages
import scanpy as sc
import os
import pandas as pd

In [2]:
import tools
import matplotlib.pyplot as plt
import scanpy.external as sce
from matplotlib.pyplot import rc_context
import seaborn as sns
import numpy as np
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests

In [3]:
#change current directory 
os.chdir('/scratch/user/uqlbukat/NRAtlas_analysis_2026/NR_code/NewDC2subsets_Analysis')
os.getcwd()

'/scratch/user/uqlbukat/NRAtlas_analysis_2026/NR_code/NewDC2subsets_Analysis'

Need to use raw unnormalised data when performing pseudobulking

In [4]:
# Read in the data 
data = sc.read_h5ad('/scratch/user/uqlbukat/NRAtlas_analysis_2026/NRclean_Raw_Data.h5ad')

In [5]:
data

AnnData object with n_obs × n_vars = 498023 × 15074
    obs: 'nCount_RNA', 'nFeature_RNA', 'sample_type', 'cancer_type', 'patient_id', 'sample_id', 'percent.mt', 'site', 'sample_type_major', 'cancer_subtype', 'integration_id', 'ident', 'dataset_id', 'Technology', 'NR_annotations_simple', 'sample_type_major2', 'cancer_with_H', 'cancer_broadest', 'patient_treatment'
    var: 'name'

Isolate the ovarian cancer data

In [6]:
data_oc = data[data.obs["cancer_with_H"] == "HGSOC"].copy()

In [7]:
data_oc

AnnData object with n_obs × n_vars = 213319 × 15074
    obs: 'nCount_RNA', 'nFeature_RNA', 'sample_type', 'cancer_type', 'patient_id', 'sample_id', 'percent.mt', 'site', 'sample_type_major', 'cancer_subtype', 'integration_id', 'ident', 'dataset_id', 'Technology', 'NR_annotations_simple', 'sample_type_major2', 'cancer_with_H', 'cancer_broadest', 'patient_treatment'
    var: 'name'

In [9]:
unique_count = data_oc.obs['sample_type_major2'].unique()
unique_count

['ascites', 'primary tumour', 'metastatic tumour', 'healthy']
Categories (4, object): ['ascites', 'healthy', 'metastatic tumour', 'primary tumour']

In [10]:
unique_count = data_oc.obs['NR_annotations_simple'].unique()
unique_count

['Tissue resident macrophage', 'KI-67+ Tissue resident macrophage', 'mregDC', 'Classical monocytes', 'non-classical monocytes', 'cDC1', 'cDC2']
Categories (7, object): ['Classical monocytes', 'KI-67+ Tissue resident macrophage', 'Tissue resident macrophage', 'cDC1', 'cDC2', 'mregDC', 'non-classical monocytes']

Isolate the DC2 data

In [11]:
data_dc2 = data_oc[data_oc.obs["NR_annotations_simple"] == "cDC2"].copy()

subset the DC2 data to only primary and metastatic

In [12]:
# Filter for multiple specific values using .isin()
cancer_stage = ['primary tumour', 'metastatic tumour']
data_sub = data_dc2[data_dc2.obs['sample_type_major2'].isin(cancer_stage)].copy()

In [13]:
unique_count = data_sub.obs['sample_type_major2'].unique()
unique_count

['primary tumour', 'metastatic tumour']
Categories (2, object): ['metastatic tumour', 'primary tumour']

## Transfer annotations from DC2 object onto the raw data

In [18]:
#read in DC2 data with Sep 2026 annotations (NR made new dataset called 20261003_DC2_bbknn_newannots.h5ad)
data_x = sc.read_h5ad('/scratch/user/uqlbukat/NRAtlas_analysis_2026/20261003_DC2_bbknn_newannots.h5ad')

In [26]:
data_x.obs

,nCount_RNA,nFeature_RNA,sample_type,cancer_type,patient_id,sample_id,percent.mt,site,sample_type_major,cancer_subtype,...,2025-DC2-true_08_sub5-1,cDC2A_sig,cDC2B_sig,cDC2_A-B,cDC2_Atlas_Subclusters,DC_CD207_Moon_sig,cDC2A_Minutti_sig,DC2_CD207_Gui_sig,DC2_CD1A_Cheng_sig,cDC2_sub_annotated
GSE215120_AM1_AAATGCCCAGAGCCAA-1,17143.000000,2659.000000,tumour,Acral Melanoma,AM1,GSE215120_Acral_MEL_AM1,1.224990,skin,primary tumour,Acral Melanoma,...,0,0.147720,0.597550,cDC2B,1,-0.285166,0.061035,-0.227976,-0.075076,DC3-like cDC2
GSE215120_AM1_ACACTGATCCACTGGG-1,18333.000000,2845.000000,tumour,Acral Melanoma,AM1,GSE215120_Acral_MEL_AM1,1.063656,skin,primary tumour,Acral Melanoma,...,0,-0.100811,0.181466,cDC2B,1,-0.263499,-0.025313,-0.229257,-0.267238,DC3-like cDC2
GSE215120_AM1_ACAGCCGCAAACCTAC-1,10581.000000,2182.000000,tumour,Acral Melanoma,AM1,GSE215120_Acral_MEL_AM1,2.050846,skin,primary tumour,Acral Melanoma,...,0,-0.221879,-0.618129,cDC2B,1,-0.391576,-0.221879,-0.295809,-0.391943,DC3-like cDC2
GSE215120_AM1_ACGTCAATCCGCATCT-1,9833.000000,2074.000000,tumour,Acral Melanoma,AM1,GSE215120_Acral_MEL_AM1,1.311909,skin,primary tumour,Acral Melanoma,...,2,-0.227921,-0.107089,cDC2B,5,-0.240897,-0.227921,-0.263913,-0.379968,classical cDC2
GSE215120_AM1_ACTTGTTTCTGAAAGA-1,7887.000000,1846.000000,tumour,Acral Melanoma,AM1,GSE215120_Acral_MEL_AM1,2.738684,skin,primary tumour,Acral Melanoma,...,2,-0.199067,0.283209,cDC2B,5,-0.106461,-0.199067,-0.207706,-0.095389,classical cDC2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSE180661_HGSOC_SPECTRUM-OV-118_S1_CD45P_LEFT_ADNEXA_TGATCAGCAGTGTGCC,-9.194493,-8.535726,Adnexa,HGSOC,Left Adnexa,SPECTRUM-OV-118_S1_CD45P_LEFT_ADNEXA,2133.000000,ovary,primary tumour,HGSOC,...,"5,0",-0.258023,-0.668658,cDC2A,8,0.284868,-0.043808,-0.361723,0.363466,CD207-IL18 cDC2
GSE180661_HGSOC_SPECTRUM-OV-118_S1_CD45P_LEFT_ADNEXA_TGCGGCACAGAGGAAA,-9.025559,-8.786784,Adnexa,HGSOC,Left Adnexa,SPECTRUM-OV-118_S1_CD45P_LEFT_ADNEXA,2122.000000,ovary,primary tumour,HGSOC,...,2,-0.262817,-0.097966,cDC2B,5,-0.457705,-0.044300,-0.274982,-0.422830,classical cDC2
GSE180661_HGSOC_SPECTRUM-OV-118_S1_CD45P_LEFT_ADNEXA_TGGAGAGAGGCTTCCG,-8.552586,-8.416667,Adnexa,HGSOC,Left Adnexa,SPECTRUM-OV-118_S1_CD45P_LEFT_ADNEXA,2330.000000,ovary,primary tumour,HGSOC,...,2,-0.298592,-0.738670,cDC2B,5,-0.320366,-0.298592,-0.242780,-0.436415,classical cDC2
GSE180661_HGSOC_SPECTRUM-OV-118_S1_CD45P_LEFT_ADNEXA_TGTTGAGCAGCAAGAC,-9.298763,-8.526468,Adnexa,HGSOC,Left Adnexa,SPECTRUM-OV-118_S1_CD45P_LEFT_ADNEXA,1333.000000,ovary,primary tumour,HGSOC,...,2,0.194285,-0.248999,cDC2B,5,0.538345,0.410091,0.676352,0.435687,classical cDC2


In [28]:
unique_count = data_x.obs['cDC2_sub_annotated'].unique()
unique_count

['DC3-like cDC2', 'classical cDC2', 'CD207-LTB cDC2', 'ISG cDC2', 'CD207-S100 cDC2', 'SIK3 cDC2', 'CD207-IL18 cDC2']
Categories (7, object): ['CD207-IL18 cDC2', 'CD207-LTB cDC2', 'CD207-S100 cDC2', 'DC3-like cDC2', 'ISG cDC2', 'SIK3 cDC2', 'classical cDC2']

Subset to primary and metastatic only

In [19]:
# Filter for multiple specific values using .isin()
cancer_stage = ['primary tumour', 'metastatic tumour']
data_x = data_x[data_x.obs['sample_type_major2'].isin(cancer_stage)].copy()


In [ ]:
#check DC2 subclusters
unique_count = data_sub.obs['sample_type_major2'].unique()
unique_count

### Align data

In [29]:
#raw object data
#data sub -> raw data
#data_x -> processed data 

data_sub.obs['cDC2'] = np.nan

common_cells = data_sub.obs_names.intersection(
    data_x.obs_names
)

data_sub.obs.loc[
    common_cells,
    'cDC2'
] = data_x.obs.loc[
    common_cells,
    'cDC2_sub_annotated'
]

/scratch/temp/30203215/ipykernel_2731845/1789776031.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '['DC3-like cDC2', 'classical cDC2', 'DC3-like cDC2', 'classical cDC2', 'DC3-like cDC2', ..., 'CD207-IL18 cDC2', 'classical cDC2', 'classical cDC2', 'classical cDC2', 'DC3-like cDC2']
Length: 2947
Categories (7, object): ['CD207-IL18 cDC2', 'CD207-LTB cDC2', 'CD207-S100 cDC2', 'DC3-like cDC2', 'ISG cDC2', 'SIK3 cDC2', 'classical cDC2']' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  data_sub.obs.loc[


In [30]:
data_sub.obs['cDC2'].value_counts(dropna=False)

cDC2
classical cDC2     1440
DC3-like cDC2       770
NaN                 310
SIK3 cDC2           305
CD207-IL18 cDC2     255
ISG cDC2            132
CD207-LTB cDC2       25
CD207-S100 cDC2      20
Name: count, dtype: int64

Check Composition

In [31]:
pd.crosstab(data_sub.obs['cDC2'], data_sub.obs['sample_type_major2'])

sample_type_major2,metastatic tumour,primary tumour
cDC2,,
CD207-IL18 cDC2,173,82
CD207-LTB cDC2,18,7
CD207-S100 cDC2,10,10
DC3-like cDC2,416,354
ISG cDC2,94,38
SIK3 cDC2,40,265
classical cDC2,923,517


# Make pseudobulk dataframes: 

In [32]:
groupby_cols = ["sample", "cDC2"]

In [35]:
import decoupler as dc


In [ ]:
#pdata = dc.get_pseudobulk(data_sub,sample_col='sample',groups_col='cDC2',layer=None,mode='sum')
#module 'decoupler' has no attribute 'get_pseudobulk'
#didn't work so using a different method to pseudobulk data

AttributeError: module 'decoupler' has no attribute 'get_pseudobulk'

In [ ]:
#subset out each DC
data_CD207_IL18 = data_sub[data_sub.obs["cDC2"] == "CD207-IL18 cDC2"]
data_CD207_LTB = data_sub[data_sub.obs["cDC2"] == "CD207-LTB cDC2"]
data_CD207_S100 = data_sub[data_sub.obs["cDC2"] == "CD207-S100 cDC2"]
data_ISG = data_sub[data_sub.obs["cDC2"] == "ISG cDC2"]
data_SIK3 = data_sub[data_sub.obs["cDC2"] == "SIK3 cDC2"]
data_dc2 = data_sub[data_sub.obs["cDC2"] == "classical cDC2"]
data_dc3 = data_sub[data_sub.obs["cDC2"] == "DC3-like cells"]

In [39]:
#checking how many samples in each (this will have to match metadata dataframe below)
num_unique_integration_ids = data_CD207_IL18.obs['integration_id'].nunique()
print(f"Number of unique integration_id values: {num_unique_integration_ids}")

Number of unique integration_id values: 63


In [40]:
data_CD207_IL18_matrix = data_CD207_IL18.X.toarray()
data_CD207_LTB_matrix = data_CD207_LTB.X.toarray()
data_CD207_S100_matrix = data_CD207_S100.X.toarray()
data_ISG_matrix = data_ISG.X.toarray()
data_SIK3_matrix = data_SIK3.X.toarray()
data_dc2_matrix = data_dc2.X.toarray()
data_dc3_matrix = data_dc3.X.toarray()

Don't need to analyse the data_CD207_LTB and data_CD207_S100 because there are too few samples in these

In [41]:
genes_IL18 = data_CD207_IL18.var_names
obs_IL18 = data_CD207_IL18.obs

genes_ISG = data_ISG.var_names
obs_ISG = data_ISG.obs

genes_SIK3 = data_SIK3.var_names
obs_SIK3 = data_SIK3.obs

genes_dc2 = data_dc2.var_names
obs_dc2 = data_dc2.obs

genes_dc3 = data_dc3.var_names
obs_dc3 = data_dc3.obs

In [42]:
# Create a DataFrame with cells as rows and genes as columns
data_IL18_df = pd.DataFrame(data_CD207_IL18_matrix, index= obs_IL18.index, columns=genes_IL18)
data_ISG_df = pd.DataFrame(data_ISG_matrix, index=obs_ISG.index, columns=genes_ISG)
data_SIK3_df = pd.DataFrame(data_SIK3_matrix, index=obs_SIK3.index, columns=genes_SIK3)
data_dc2_df = pd.DataFrame(data_dc2_matrix, index=obs_dc2.index, columns=genes_dc2)
data_dc3_df = pd.DataFrame(data_dc3_matrix, index=obs_dc3.index, columns=genes_dc3)

In [43]:
# Add integration_id
data_IL18_df['integration_id'] = obs_IL18['integration_id']
data_ISG_df['integration_id'] = obs_ISG['integration_id']
data_SIK3_df['integration_id'] = obs_SIK3['integration_id']
data_dc2_df['integration_id'] = obs_dc2['integration_id']
data_dc3_df['integration_id'] = obs_dc3['integration_id']

In [44]:
data_IL18_df.head()

,FAM87B,LINC00115,FAM41C,SAMD11,NOC2L,KLHL17,PLEKHN1,HES4,ISG15,AGRN,...,MT-ATP8,MT-ATP6,MT-CO3,MT-ND3,MT-ND4L,MT-ND4,MT-ND5,MT-ND6,MT-CYB,integration_id
GSE184880_Cancer1_GGGTCACCAAGCTGTT-1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,2.0,104.0,65.0,52.0,6.0,74.0,14.0,0.0,56.0,GSE184880_HGSOC_Cancer1
GSE184880_Cancer7_CCGTTCATCTGAACGT-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,...,3.0,61.0,71.0,53.0,5.0,80.0,19.0,1.0,39.0,GSE184880_HGSOC_Cancer7
GSE184880_Cancer7_GCAGCCAAGCATGCAG-1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,60.0,63.0,67.0,6.0,47.0,10.0,0.0,17.0,GSE184880_HGSOC_Cancer7
GSE217517_pt2_ACACAGTGTTCTCGCT-1,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,10.0,0.0,...,11.0,49.0,54.0,41.0,5.0,33.0,13.0,0.0,30.0,GSE217517_HGSOC_pt2
GSE217517_pt2_TTATTGCGTTACCCAA-1,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,3.0,1.0,...,24.0,186.0,144.0,124.0,8.0,62.0,11.0,1.0,85.0,GSE217517_HGSOC_pt2


In [45]:
# Group by integration_id and sum to pseudobulk
pseudobulk_IL18_df = data_IL18_df.groupby('integration_id').sum().transpose()
pseudobulk_ISG_df = data_ISG_df.groupby('integration_id').sum().transpose()
pseudobulk_SIK3_df = data_SIK3_df.groupby('integration_id').sum().transpose()
pseudobulk_dc2_df = data_dc2_df.groupby('integration_id').sum().transpose()
pseudobulk_dc3_df = data_dc3_df.groupby('integration_id').sum().transpose()

/scratch/temp/30203215/ipykernel_2731845/2541314046.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pseudobulk_IL18_df = data_IL18_df.groupby('integration_id').sum().transpose()
/scratch/temp/30203215/ipykernel_2731845/2541314046.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  pseudobulk_ISG_df = data_ISG_df.groupby('integration_id').sum().transpose()
/scratch/temp/30203215/ipykernel_2731845/2541314046.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the f

In [46]:
pseudobulk_IL18_df.head()

integration_id,GSE180661_SPECTRUM-OV-003_S1_UNSORTED_LEFT_UPPER_QUADRANT,GSE180661_SPECTRUM-OV-009_S1_CD45P_BOWEL,GSE180661_SPECTRUM-OV-009_S1_CD45P_INFRACOLIC_OMENTUM,GSE180661_SPECTRUM-OV-009_S1_CD45P_PELVIC_PERITONEUM,GSE180661_SPECTRUM-OV-009_S1_CD45P_RIGHT_UPPER_QUADRANT,GSE180661_SPECTRUM-OV-014_S1_CD45P_BOWEL,GSE180661_SPECTRUM-OV-014_S1_CD45P_LEFT_OVARY,GSE180661_SPECTRUM-OV-014_S1_CD45P_PELVIC_PERITONEUM,GSE180661_SPECTRUM-OV-022_S1_CD45P_LEFT_ADNEXA,GSE180661_SPECTRUM-OV-025_S1_CD45P_BOWEL,...,GSE180661_SPECTRUM-OV-115_S1_CD45P_INFRACOLIC_OMENTUM,GSE180661_SPECTRUM-OV-115_S1_CD45P_LEFT_ADNEXA,GSE180661_SPECTRUM-OV-115_S1_CD45P_PELVIC_PERITONEUM,GSE180661_SPECTRUM-OV-115_S1_CD45P_RIGHT_ADNEXA,GSE180661_SPECTRUM-OV-118_S1_CD45P_LEFT_ADNEXA,GSE180661_SPECTRUM-OV-118_S1_CD45P_PELVIC_PERITONEUM,GSE180661_SPECTRUM-OV-118_S1_CD45P_RIGHT_ADNEXA,GSE184880_HGSOC_Cancer1,GSE184880_HGSOC_Cancer7,GSE217517_HGSOC_pt2
FAM87B,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LINC00115,0.0,1.0,1.0,0.0,0.0,1.0,2.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
FAM41C,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
SAMD11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
NOC2L,1.0,2.0,0.0,0.0,1.0,8.0,7.0,5.0,1.0,0.0,...,2.0,0.0,2.0,2.0,2.0,0.0,1.0,1.0,0.0,3.0


In [47]:
pseudobulk_SIK3_df.head()

integration_id,GSE180661_SPECTRUM-OV-014_S1_CD45P_BOWEL,GSE180661_SPECTRUM-OV-025_S1_CD45P_BOWEL,GSE180661_SPECTRUM-OV-025_S1_CD45P_INFRACOLIC_OMENTUM,GSE180661_SPECTRUM-OV-025_S1_CD45P_RIGHT_OVARY,GSE180661_SPECTRUM-OV-026_S1_CD45P_BOWEL,GSE180661_SPECTRUM-OV-026_S1_CD45P_LEFT_ADNEXA,GSE180661_SPECTRUM-OV-031_S1_CD45P_LEFT_FALLOPIAN_TUBE,GSE180661_SPECTRUM-OV-036_S1_CD45P_RIGHT_ADNEXA,GSE180661_SPECTRUM-OV-037_S1_CD45P_INFRACOLIC_OMENTUM,GSE180661_SPECTRUM-OV-037_S1_CD45P_LEFT_OVARY,...,GSE180661_SPECTRUM-OV-107_S1_CD45P_INFRACOLIC_OMENTUM,GSE180661_SPECTRUM-OV-107_S1_CD45P_RIGHT_ADNEXA,GSE180661_SPECTRUM-OV-112_S1_CD45P_LEFT_ADNEXA,GSE184880_HGSOC_Cancer1,GSE217517_HGSOC_pt1,GSE217517_HGSOC_pt2,GSE217517_HGSOC_pt3,GSE217517_HGSOC_pt4,GSE217517_HGSOC_pt5,GSE217517_HGSOC_pt7
FAM87B,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
LINC00115,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,4.0,8.0,3.0,0.0,0.0,1.0
FAM41C,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,3.0,5.0,0.0,0.0,0.0,0.0
SAMD11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
NOC2L,1.0,0.0,0.0,0.0,0.0,2.0,2.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,14.0,34.0,8.0,0.0,0.0,3.0


In [48]:
pseudobulk_ISG_df.head()

integration_id,GSE180661_SPECTRUM-OV-002_S1_CD45P_RIGHT_OVARY,GSE180661_SPECTRUM-OV-003_S1_UNSORTED_LEFT_UPPER_QUADRANT,GSE180661_SPECTRUM-OV-008_S1_CD45P_INFRACOLIC_OMENTUM,GSE180661_SPECTRUM-OV-009_S1_CD45P_BOWEL,GSE180661_SPECTRUM-OV-009_S1_CD45P_INFRACOLIC_OMENTUM,GSE180661_SPECTRUM-OV-009_S1_CD45P_LEFT_UPPER_QUADRANT,GSE180661_SPECTRUM-OV-014_S1_CD45P_BOWEL,GSE180661_SPECTRUM-OV-014_S1_CD45P_LEFT_OVARY,GSE180661_SPECTRUM-OV-014_S1_CD45P_PELVIC_PERITONEUM,GSE180661_SPECTRUM-OV-022_S1_CD45P_BOWEL,...,GSE180661_SPECTRUM-OV-107_S1_CD45P_RIGHT_ADNEXA,GSE180661_SPECTRUM-OV-110_S1_CD45P_INFRACOLIC_OMENTUM,GSE180661_SPECTRUM-OV-115_S1_CD45P_INFRACOLIC_OMENTUM,GSE180661_SPECTRUM-OV-115_S1_CD45P_PELVIC_PERITONEUM,GSE180661_SPECTRUM-OV-116_S1_CD45P_INFRACOLIC_OMENTUM,GSE180661_SPECTRUM-OV-118_S1_CD45P_LEFT_ADNEXA,GSE180661_SPECTRUM-OV-118_S1_CD45P_PELVIC_PERITONEUM,GSE180661_SPECTRUM-OV-118_S1_CD45P_RIGHT_ADNEXA,GSE184880_HGSOC_Cancer2,GSE184880_HGSOC_Cancer7
FAM87B,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
LINC00115,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0
FAM41C,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
SAMD11,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
NOC2L,0.0,0.0,0.0,0.0,1.0,1.0,0.0,2.0,4.0,0.0,...,0.0,0.0,1.0,1.0,1.0,0.0,3.0,0.0,4.0,0.0


In [49]:
os.getcwd()

'/scratch/user/uqlbukat/NRAtlas_analysis_2026/NR_code/NewDC2subsets_Analysis'

In [50]:
os.chdir('/scratch/user/uqlbukat/NRAtlas_analysis_2026/NR_code/NewDC2subsets_Analysis/DC2_Pseudobulk')

In [51]:
# save as csv
pseudobulk_IL18_df.to_csv("pseudobulk_IL18_data.csv", index=True)
pseudobulk_ISG_df.to_csv("pseudobulk_ISG_data.csv", index=True)
pseudobulk_SIK3_df.to_csv("pseudobulk_SIK3_data.csv", index=True)
pseudobulk_dc2_df.to_csv("pseudobulk_dc2_data.csv", index=True)
pseudobulk_dc3_df.to_csv("pseudobulk_dc3_data.csv", index=True)

# Make metadata dataframes

In [52]:
data_CD207_IL18

View of AnnData object with n_obs × n_vars = 255 × 15074
    obs: 'nCount_RNA', 'nFeature_RNA', 'sample_type', 'cancer_type', 'patient_id', 'sample_id', 'percent.mt', 'site', 'sample_type_major', 'cancer_subtype', 'integration_id', 'ident', 'dataset_id', 'Technology', 'NR_annotations_simple', 'sample_type_major2', 'cancer_with_H', 'cancer_broadest', 'patient_treatment', 'cDC2'
    var: 'name'

In [53]:
metadata_IL18 = data_CD207_IL18.obs[['integration_id', 'sample_type_major2']].groupby('integration_id').first()
metadata_ISG = data_ISG.obs[['integration_id', 'sample_type_major2']].groupby('integration_id').first()
metadata_SIK3 = data_SIK3.obs[['integration_id', 'sample_type_major2']].groupby('integration_id').first()
metadata_dc2 = data_dc2.obs[['integration_id', 'sample_type_major2']].groupby('integration_id').first()
metadata_dc3 = data_dc3.obs[['integration_id', 'sample_type_major2']].groupby('integration_id').first()


/scratch/temp/30203215/ipykernel_2731845/1116243768.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  metadata_IL18 = data_CD207_IL18.obs[['integration_id', 'sample_type_major2']].groupby('integration_id').first()
/scratch/temp/30203215/ipykernel_2731845/1116243768.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  metadata_ISG = data_ISG.obs[['integration_id', 'sample_type_major2']].groupby('integration_id').first()
/scratch/temp/30203215/ipykernel_2731845/1116243768.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=Fa

In [54]:
metadata_IL18.head()

,sample_type_major2
integration_id,
GSE180661_SPECTRUM-OV-003_S1_UNSORTED_LEFT_UPPER_QUADRANT,metastatic tumour
GSE180661_SPECTRUM-OV-009_S1_CD45P_BOWEL,metastatic tumour
GSE180661_SPECTRUM-OV-009_S1_CD45P_INFRACOLIC_OMENTUM,metastatic tumour
GSE180661_SPECTRUM-OV-009_S1_CD45P_PELVIC_PERITONEUM,metastatic tumour
GSE180661_SPECTRUM-OV-009_S1_CD45P_RIGHT_UPPER_QUADRANT,metastatic tumour


In [55]:
metadata_IL18.shape

(63, 1)

In [56]:
# set the index (integration_id) to be a column
metadata_IL18 = metadata_IL18.reset_index()
metadata_ISG = metadata_ISG.reset_index()
metadata_SIK3 = metadata_SIK3.reset_index()
metadata_dc2 = metadata_dc2.reset_index()
metadata_dc3 = metadata_dc3.reset_index()

In [57]:
metadata_IL18.head()

,integration_id,sample_type_major2
0,GSE180661_SPECTRUM-OV-003_S1_UNSORTED_LEFT_UPP...,metastatic tumour
1,GSE180661_SPECTRUM-OV-009_S1_CD45P_BOWEL,metastatic tumour
2,GSE180661_SPECTRUM-OV-009_S1_CD45P_INFRACOLIC_...,metastatic tumour
3,GSE180661_SPECTRUM-OV-009_S1_CD45P_PELVIC_PERI...,metastatic tumour
4,GSE180661_SPECTRUM-OV-009_S1_CD45P_RIGHT_UPPER...,metastatic tumour


In [58]:
os.getcwd()

'/scratch/user/uqlbukat/NRAtlas_analysis_2026/NR_code/NewDC2subsets_Analysis/DC2_Pseudobulk'

In [59]:
# save as csv
metadata_IL18.to_csv("pseudobulk_metadata_IL18.csv", index=False)
metadata_ISG.to_csv("pseudobulk_metadata_ISG.csv", index=False)
metadata_SIK3.to_csv("pseudobulk_metadata_SIK3.csv", index=False)
metadata_dc2.to_csv("pseudobulk_metadata_dc2.csv", index=False)
metadata_dc3.to_csv("pseudobulk_metadata_dc3.csv", index=False)